# Agent和AgentExecutor的创建

- Agent（智能体/大脑）：它只负责“思考”和“决策”。它接收用户的输入，看看手里有哪些工具，然后决定下一步该调哪个工具。它绝对不会自己去执行代码！


- AgentExecutor（执行器/肉体）：它是底层的运行时（Runtime） 。它负责拿着 Agent 做的决定，去本地真实地调用 Python 函数（工具），然后把执行结果拿回来，再喂给 Agent 继续思考 。



  
   
<hr>

| 对比维度         | 方式 1：传统方式 (initialize_agent)                        | 方式 2：通用方式 (create_xxx_agent)               |
|------------------|----------------------------------------------------------|--------------------------------------------------|
| 设计哲学         | 黑盒封装：一个函数包揽一切。                              | 积木解耦：Agent 负责想，Executor 负责做。        |
| Agent 的指定     | 传枚举值：agent=AgentType.XXX +2                           | 调专属函数：create_react_agent() 等 +2            |
| 提示词 (Prompt)  | 框架底层写死的，你看不见也改不了。                        | 强制要求自定义，你可以随意修改系统人设。 +1        |
| 适用场景         | 快速写个 Demo 跑着玩。                                   | 真实的商业生产环境，需要高度可控和排错。           |

<hr>

| 对比维度           | 🧠 ReAct 模式 (文本推理)                                     | ⚙️ Function Call 模式 (结构化调用)               |
|--------------------|-----------------------------------------------------------|-----------------------------------------------|
| 底层机制           | 基于自然语言推理。模型生成文本指令，系统解析后再调用工具。+2 | 基于结构化函数调用。模型直接返回 JSON 格式的工具调用指令。+2 |
| 输出格式           | 自由文本（需要遵循严格的 Thought/Action/Observation 格式）。+3 | JSON 或结构化数据。+2                          |
| 执行效率与延迟     | 较低/延迟高。因为需要多轮文本交互，且需生成完整的解释性文本。+2 | 更高/延迟低。单步完成，直接进行参数化调用。+2    |
| 输出可读性         | 极佳。直接显示人类可读的思考过程，能看懂 AI 为什么这么做。 | 较差。中间过程是机器语言，需要查看结构化日志才能看懂。 |
| 工具参数处理       | 依赖模型文本描述的准确性，有几率因为文本解析失败而出错。+1 | 自动匹配工具的参数结构，提供结构化保证，更加可靠。 |
| 对大模型的要求     | 门槛低。所有通用的文本生成模型即可支持。+2                 | 门槛高。需要大模型在底层专门经过函数调用的微调训练（如 GPT-4、Claude 3 等新模型）。+2 |
| 典型 AgentType     | ZERO_SHOT_REACT_DESCRIPTION 等。                         | OPENAI_FUNCTIONS、OPENAI_MULTI_FUNCTIONS 等。+1  |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool

# 🕰️ 传统方式的核心导入
from langchain.agents import initialize_agent, AgentType

import os
import dotenv
from langchain_openai import ChatOpenAI

# 1. 基础配置
# 2. 定义严格的大模型 (调用工具温度务必设为 0)
dotenv.load_dotenv(
    # 在工程根目录只保留一个 .env，所有模块都引用它。这是最不容易出错的方式。
    override=True,  # 如果加载.env文件出现了同名环境变脸，会进行覆盖
    # dotenv_path=".env",  # 可以只当需要加载的路径
)

llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
)


@tool
def search_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city}今天是晴天。"


tools = [search_weather]

# ==============================================================================
# 📦 【传统方式】环节 1 & 环节 2 同时进行
#
# 痛点：虽然只要一行代码，但你根本看不到底层的 Prompt 是什么样子的（内置不可见）。
# ==============================================================================
legacy_executor = initialize_agent(
    tools=tools,
    llm=llm,
    # 🌟 传统方式的标志：使用 AgentType 枚举来指定环节 1（创建哪种类型的 Agent）
    # - AgentType.OPENAI_FUNCTIONS：代表创建一个 Function Call 模式的 Agent [cite: 88]。
    # - AgentType.ZERO_SHOT_REACT_DESCRIPTION：代表创建一个 ReAct 模式的 Agent [cite: 112]。
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
)

# 直接运行执行器
legacy_executor.invoke({"input": "北京天气"})

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI

# ==============================================================================
# 🧠 步骤 1：自定义极度透明的 Prompt (替代旧版的黑盒)
# ==============================================================================
# 1. 基础配置
# 2. 定义严格的大模型 (调用工具温度务必设为 0)
dotenv.load_dotenv(
    # 在工程根目录只保留一个 .env，所有模块都引用它。这是最不容易出错的方式。
    override=True,  # 如果加载.env文件出现了同名环境变脸，会进行覆盖
    # dotenv_path=".env",  # 可以只当需要加载的路径
)

llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位超级AI特工。请尽可能使用工具来解答用户问题。"),
        ("human", "{input}"),
        # 🚨 极度易错参数：agent_scratchpad (特工草稿本)
        # 作用: 它是必须声明的占位符 。
        # 机制: 当 Agent 调用了工具后，工具返回的结果（Observation）以及特工的中间思考过程，
        # 都会被格式化并塞进这个变量里，避免上下文丢失。如果不写，100% 报错 KeyError 。
        ("placeholder", "{agent_scratchpad}"),
    ]
)

# ==============================================================================
# 🤖 步骤 2：单独创建“思考大脑” (Agent)
#
# 机制: 现代 API 使用 create_xxx_agent 来明确模式 [cite: 2203]。
# - create_tool_calling_agent: 代表使用现代的 Function Call (bind_tools) 模式 [cite: 2124]。
# - create_react_agent: 代表使用文本推理的 ReAct 模式 [cite: 2124]。
# ==============================================================================
agent = create_tool_calling_agent(
    llm=llm,  # 绑定模型 [cite: 2895]
    tools=tools,  # 告诉大脑有哪些工具可用 [cite: 2897]
    prompt=prompt,  # 注入你刚刚写的灵魂提示词 [cite: 2899]
)

# ==============================================================================
# ⚙️ 步骤 3：单独创建“运行时执行器” (AgentExecutor)
#
# 作用: AgentExecutor 本质上是代理的运行时，负责协调智能体的决策和实际的工具执行 [cite: 2120]。
# ==============================================================================
modern_agent_executor = AgentExecutor(
    # 参数: agent
    # 作用: 把刚刚创建的“大脑”塞进执行器 [cite: 2904]。
    agent=agent,
    # 参数: tools
    # 作用: 再次提供工具。因为执行器运行在本地，它需要真正的 Python 函数来执行大模型的指令 [cite: 2904]。
    tools=tools,
    verbose=True,
    # 🛡️ 生产环境救命稻草：handle_parsing_errors
    # 作用: 控制 Agent 在解析大模型输出失败时的容错行为 [cite: 3252]。
    # 机制: 当大模型“发神经”输出乱码而不是 JSON 或标准 Action 格式时，设为 True 会自动捕获错误，
    # 并将错误信息发回给大模型让它“自我修复并重试”，而不是直接让程序死机崩溃 [cite: 3254, 3255, 3328]！
    handle_parsing_errors=True,
    # ⏱️ 参数: max_iterations (笔记中提到的防死循环参数)
    # 作用: 限制大模型最多只能连续思考和调用工具的次数（比如 6 次），防止大模型陷入“死胡同”狂刷 API 费用 [cite: 3233]。
    max_iterations=6,
)

print("\n=== 🚀 运行现代 AgentExecutor ===")
result = modern_agent_executor.invoke({"input": "北京天气怎么样？"})
print(f"\n🎯 最终回答: {result['output']}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 🚀 通用方式的核心导入：明确区分 Agent 和 Executor
from langchain.agents import create_tool_calling_agent, AgentExecutor

import os
import dotenv
from langchain_openai import ChatOpenAI

# 1. 基础配置
# 2. 定义严格的大模型 (调用工具温度务必设为 0)
dotenv.load_dotenv(
    # 在工程根目录只保留一个 .env，所有模块都引用它。这是最不容易出错的方式。
    override=True,  # 如果加载.env文件出现了同名环境变脸，会进行覆盖
    # dotenv_path=".env",  # 可以只当需要加载的路径
)

llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
)

# ==============================================================================
# 🧠 【通用方式】环节 1：显式构建 Agent (大脑)
# ==============================================================================

# 第 1 步：由于是解耦的，你必须亲自给大脑注入灵魂（Prompt）
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个天气预报员，请使用工具回答问题。"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),  # 必须保留的思考草稿本 [cite: 831]
    ]
)


@tool
def search_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city}今天是晴天。"


tools = [search_weather]

# 第 2 步：使用专门的 create_xxx_agent 方法创建大脑
# 这里的 create_tool_calling_agent 就等价于传统方式里的 AgentType.OPENAI_FUNCTIONS
agent_brain = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
# ⚠️ 注意：此时的 agent_brain 只是一个思考逻辑，它无法直接运行工具！

# ==============================================================================
# ⚙️ 【通用方式】环节 2：显式构建 AgentExecutor (执行器/肉体) [cite: 177, 187]
# ==============================================================================

# 第 3 步：把刚刚创建的“大脑 (agent_brain)”装进“肉体 (AgentExecutor)”里 [cite: 179]
modern_executor = AgentExecutor(
    agent=agent_brain,  # 注入大脑 [cite: 179]
    tools=tools,  # 再次提供工具，因为肉体需要真正在本地调用这些 Python 函数 [cite: 181]
    verbose=True,
    handle_parsing_errors=True,  # 开启生产环境的防崩溃自动修复功能 [cite: 1204]
)

# 真正运行的是 Executor
print("=== 🚀 通用方式执行 ===")
result = modern_executor.invoke({"input": "北京天气"})
print(result)

# 1、单工具的使用

## 使用ReAct模型

In [ ]:
import os
import dotenv

# 导入 Tavily 搜索引擎（专为 Agent 设计的无广告、结构化搜索引擎）
from langchain_community.tools.tavily_search import TavilySearchResults

# 导入 LangChain 的标准工具封装类
from langchain.tools import Tool

# 导入 OpenAI/兼容大模型接口
from langchain_openai import ChatOpenAI

# 导入 LangChain Hub（类似于 Docker Hub，专门用来拉取云端的优秀 Prompt 模板）
from langchain import hub

# 导入现代版 Agent 创建函数和物理执行器
from langchain.agents import create_react_agent, AgentExecutor

# ==============================================================================
# ⚙️ 准备工作：环境变量加载
# ==============================================================================
dotenv.load_dotenv(override=True)
# 确保在 .env 文件中已经配置了 TAVILY_API_KEY，否则工具无法联网
# 防止你的 API 请求被本地的代理软件（如 Clash、VPN）拦截而导致网络连接报错（SSLError 等）。
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
# 1. 基础配置
# 2. 定义严格的大模型 (调用工具温度务必设为 0)
dotenv.load_dotenv(
    # 在工程根目录只保留一个 .env，所有模块都引用它。这是最不容易出错的方式。
    override=True,  # 如果加载.env文件出现了同名环境变脸，会进行覆盖
    # dotenv_path=".env",  # 可以只当需要加载的路径
)

# ==============================================================================
# 🔍 步骤 1：获取 Tavily 搜索工具的实例
# ==============================================================================
# 参数: max_results=3
# 作用: 限制搜索引擎每次最多返回 3 条网页摘要。
# 机制: 防止搜索结果太长导致大模型的 Token 数量超载（也就是爆显存）。
tavily_search = TavilySearchResults(max_results=3)

# ==============================================================================
# 🛠️ 步骤 2：获取一个标准的搜索工具 (Tool)
# ==============================================================================
# 机制: 大模型不认识 TavilySearchResults 对象，我们必须用 Tool 类把它“包装”成标准工具。
search_tool = Tool(
    # 参数: name (工具名称)
    # 作用: 大模型在输出 Action 时，必须严格原样输出这个名字。
    name="Search",
    # 参数: func (执行函数)
    # 作用: 告诉底层代码，当大模型决定用这个工具时，去执行哪个具体的 Python 函数。
    func=tavily_search.run,
    # 参数: description (工具描述) 🌟【极度重要】
    # 作用: 这绝对不是写给程序员看的注释！这是直接发给大模型看的“工具使用说明书”。
    # 机制: 大模型会阅读这段话，来判断当前用户的问题适不适合用这个工具。
    description="用于检索互联网上的信息，尤其是实时新闻、天气情况或未知的客观事实。",
)

# 构建工具箱（虽然现在只有1个工具，但未来你可以塞进去计算器、日历等多个工具）
tools = [search_tool]

# ==============================================================================
# 🧠 步骤 3：获取大语言模型 (LLM)
# ==============================================================================
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
    # 参数: temperature=0 🌟【ReAct 模式铁律】
    # 作用: 消除大模型的“想象力和创造力”。
    # 机制: ReAct 模式要求大模型必须严格按照极其死板的格式输出文字，一旦它发散思维随便乱说，
    #       底层的正则提取器就会崩溃。所以必须设为 0，让它变成严谨的机器！
    temperature=0,
)

# ==============================================================================
# 🧩 步骤 4：获取 Agent (创建思考大脑)
# ==============================================================================
# 参数: "hwchase17/react"
# 作用: 从云端拉取一套官方写好的、极其严苛的纯英文 Prompt 模板。
# 机制: 这个模板里写满了规则，强制大模型必须按照以下格式思考：
#       Thought: (我现在的想法是什么)
#       Action: (我要用什么工具，必须从 [Search, Calculator] 中选)
#       Action Input: (我要传给工具的参数是什么)
prompt = hub.pull("hwchase17/react")
print("prompt", prompt)
# 调用专属工厂函数，将 大模型、工具箱说明书、思考规则模板 揉合在一起，诞生“思考大脑”
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# ==============================================================================
# ⚙️ 步骤 5：获取 AgentExecutor 的实例 (创建物理执行器)
# ==============================================================================
# 机制: Agent 只负责“想”，AgentExecutor 负责拿着想法去“做”（在本地运行代码）。
agent_executor = AgentExecutor(
    agent=agent,  # 塞入刚刚创建的大脑
    tools=tools,  # 再次塞入工具箱（因为此时需要在本地真正运行那个 func 函数）
    # 参数: verbose=True
    # 作用: 开启上帝视角。你将在控制台看到绿色的 Thought 和蓝色的 Action，非常震撼。
    verbose=True,
    # 参数: handle_parsing_errors=True 🌟【生产环境救命稻草】
    # 作用: 自动纠错与重试机制。
    # 机制: 在 ReAct 模式下，如果大模型忘了写 "Action:" 这几个字，系统会报错。
    #       设为 True 后，系统不会崩溃，而是会自动把报错信息扔回给大模型：“你刚才输出的格式不对，请重写！”
    handle_parsing_errors=True,
    # 参数: max_iterations=6
    # 作用: 限制大模型最多只能连续进行 6 次“思考-行动”循环。
    # 机制: 防止大模型遇到解不开的难题时，陷入“无限死循环搜索”，把你的 API 余额瞬间刷光。
    max_iterations=6,
)

# ==============================================================================
# 🚀 步骤 6：通过 AgentExecutor 发起真实的调用
# ==============================================================================
print("\n🚀 正在唤醒 ReAct Agent 特工...\n")

# 注意: invoke 必须接收一个字典，其中 "input" 这个 Key 对应用户的提问。
result = agent_executor.invoke(
    {"input": "请帮我查一下昨天关于 LangChain 的最新新闻是什么？"}
)

print("\n" + "=" * 50)
# 提取出最后生成的最终回复
print(f"🎯 最终总结结果：\n{result['output']}")
print("=" * 50)

## 使用Function_Calling

In [ ]:
import os
import dotenv

# 导入 Tavily 搜索引擎
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import Tool
from langchain_openai import ChatOpenAI

# 🌟 【核心变化 1】导入现代版的 Prompt 模板类，不再使用 hub 去拉取那一大串英文规则了
from langchain_core.prompts import ChatPromptTemplate

# 🌟 【核心变化 2】导入 create_tool_calling_agent（专门用于 Function Calling）
from langchain.agents import create_tool_calling_agent, AgentExecutor

# ==============================================================================
# ⚙️ 准备工作：环境变量与工具初始化 (与 ReAct 完全一样)
# ==============================================================================
dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

tavily_search = TavilySearchResults(max_results=3)

search_tool = Tool(
    name="Search",
    func=tavily_search.run,
    description="用于检索互联网上的信息，尤其是实时新闻、天气情况或未知的客观事实。"
)

tools = [search_tool]

# ==============================================================================
# 🧠 步骤 3：获取大语言模型 (LLM)
# ==============================================================================
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B", 
    temperature=0  # 同样保持严谨
)

# ==============================================================================
# 🧩 步骤 4：获取 Agent (创建思考大脑) - 🌟 Function Calling 的高光时刻
# ==============================================================================
# 机制: Function Calling 不需要告诉大模型 "Thought/Action" 这些复杂规则，
#       因为它底层原生就懂 JSON！我们只需要写一个最简单的对话模板即可。
prompt = ChatPromptTemplate.from_messages([
    # 系统人设：随便你怎么定义，大模型自己知道什么时候该用工具
    ("system", "你是一位全知全能的超级AI助手。请尽可能使用提供的工具来解答用户的问题。"),
    
    # 接收用户的真实提问
    ("human", "{input}"),
    
    # 🚨 极度易错点：agent_scratchpad (特工草稿本)
    # 作用: 必须要有这一行！当大模型调用完工具后，底层的 AgentExecutor 会把工具的返回结果
    #       变成特定的 ToolMessage 塞进这个变量里。如果不写，大模型就会“失忆”。
    ("placeholder", "{agent_scratchpad}"),
])

print("👀 查看 Function Calling 的极简 Prompt:\n", prompt)

# 🌟 核心方法变化：使用 create_tool_calling_agent
# 底层逻辑：它会自动调用 llm.bind_tools(tools)，将你的工具翻译成 OpenAI 官方的 Function JSON 规范发给大模型。
agent = create_tool_calling_agent(
    llm=llm, 
    tools=tools, 
    prompt=prompt
)

# ==============================================================================
# ⚙️ 步骤 5：获取 AgentExecutor 的实例 (与 ReAct 基本一致)
# ==============================================================================
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, # 开启后，你会看到大模型不再输出 Thought，而是直接发起 Invoking(工具调用)
    
    # 容错处理：虽然 Function Calling 极少格式错误，但加上更保险
    handle_parsing_errors=True,
    max_iterations=6,
)

# ==============================================================================
# 🚀 步骤 6：通过 AgentExecutor 发起真实的调用
# ==============================================================================
print("\n🚀 正在唤醒 Function Calling 特工...\n")

result = agent_executor.invoke(
    {"input": "请帮我查一下昨天关于 LangChain 的最新新闻是什么？"}
)

print("\n" + "=" * 50)
print(f"🎯 最终总结结果：\n{result['output']}")
print("=" * 50)

# 📘 LangChain 高阶学习笔记：Agent 核心执行模式深度剖析 (ReAct vs Function Calling)

## 0. 背景引入：为什么需要 Agent？
在 LangChain 的基础组件中，`Chain`（链）的行动序列是硬编码的、固定流程的，像是“线性流水线”。而 `Agent`（智能体）则代表了更高的自动化级别，在 AI 等级分类中属于 L4 级别。它能够自主拆解任务、选择工具并控制进度，构建智能体(Agent)已被视为 AI 工程应用当下的“终极形态”。

目前，Agent 驱动工具执行的核心模式主要分为两大流派：**ReAct 模式** 与 **Function Calling 模式**。

---

## 🧐 一、 ReAct 模式 (Reasoning and Acting)

ReAct 是一种依赖大模型“自然语言生成能力”的早期经典 Agent 架构，它强迫模型将**推理（Reasoning）**与**行动（Acting）**结合起来。

### 1.1 核心认知机制：显式文本推理 (Explicit Textual Reasoning)
* **驱动原理**：完全依赖 **Prompt 工程**。系统通过极其严苛的纯英文提示词（如 `hwchase17/react` 模板），规定大模型必须以特定的文本格式进行“自言自语”。
* **标准思考流**：
  1. `Thought`（思考）：分析当前状况及用户需求。
  2. `Action`（行动）：决定从可用工具白名单中调用哪一个。
  3. `Action Input`（参数）：给出调用该工具所需的具体参数。
  4. `Observation`（观察）：系统执行工具后，将结果注入此环节，供模型开启下一轮 Thought。

### 1.2 物理执行流：基于“正则解析”的脆弱流水线
在 LangChain 的 `AgentExecutor` 底层，ReAct 的执行流是一个“文本切割”过程：
1. **生成与拦截**：大模型输出大段文本，LangChain 持续监听。
2. **正则提取（高危节点）**：底层使用正则表达式（Regex）去扫描文本，提取 `Action:` 和 `Action Input:` 后的字符串。
3. **反射执行**：将字符串参数传入本地 Python 函数执行。
4. **状态拼接**：将执行结果加上 `Observation: ` 前缀，强行拼接回 `agent_scratchpad`（特工草稿本）的历史文本中，再次请求大模型。

### 1.3 优缺点分析
* **优点**：**极致的可解释性**。开发人员可以直接阅读大模型的 `Thought`，清晰掌握其逻辑链路，方便调试。
* **缺点**：
  * **高延迟与高成本**：生成大量过程废话，极其消耗 Token。
  * **稳定性极差**：一旦大模型漏写标点或改变了排版，正则表达式就会崩溃。必须依赖 `handle_parsing_errors=True` 进行容错兜底。

---

## ⚙️ 二、 Function Calling 模式 (原生工具调用)

Function Calling 是由 OpenAI 等头部大模型厂商直接在**底层 API 协议**中原生支持的架构，是目前企业级生产环境的绝对主力。

### 2.1 核心认知机制：隐式结构化决策 (Implicit Neural Decision)
* **驱动原理**：**原生微调驱动 (Native Fine-tuning)**。大模型在预训练或 SFT（监督微调）阶段，已经学习了海量的“指令-JSON”映射关系，将其化为了神经网络的本能。
* **参数传递**：不再将工具说明书硬塞进 Prompt。而是通过 `bind_tools()` 将 Python 函数转化为标准的 **JSON Schema**，通过 API 的专用字段隐式传给大模型。
* **思考表现**：“黑盒思考”。大模型对外不再废话，内部完成逻辑闭环后，直接抛出一个填好参数的结构化 JSON 数据包。

### 2.2 物理执行流：基于“API 协议”的工业级流水线
1. **意图触发**：大模型生成响应，底层 API 直接在响应体中返回 `tool_calls` 列表（支持并发调用多个工具）。
2. **字典反序列化**：LangChain 直接提取出完美的 JSON 字典，**彻底告别正则表达式**。
3. **本地执行**：直接将字典拆解为 `**kwargs` 传入 Python 函数。
4. **对账闭环（核心创新）**：LangChain 将执行结果封装为独立的 `ToolMessage` 对象，并强制绑定大模型生成的 **`tool_call_id`（任务流水号）**。大模型通过核对 ID 来确认任务完成。

### 2.3 优缺点分析
* **优点**：
  * **极高稳定性**：纯 JSON 结构体交互，基本杜绝了解析崩溃。
  * **低延迟省计费**：无需输出冗长的思考过程文本，首字响应极快。
* **缺点**：中间推理过程为黑盒，排查“模型为何选错工具”时难度略大于 ReAct 模式。

---

## 📊 三、 核心差异对比矩阵

为了更直观地进行架构选型，以下是两者的核心技术维度对比：

| 评估维度 | 🧠 ReAct 模式 (文本推理) | ⚙️ Function Calling (原生调用) |
| :--- | :--- | :--- |
| **底层实现依赖** | Prompt Engineering (提示词工程) | Native Model Fine-tuning (原生模型微调) |
| **大模型行为表现** | 喋喋不休地输出 `Thought` 思考过程 | 沉默寡言，直接抛出 `JSON` 参数包 |
| **代码框架通信介质**| 纯文本 (String) | 结构化对象 (JSON / Dict) |
| **Agent 解析机制** | 正则表达式 (Regex) 字符串提取 | API 字段原生反序列化读取 |
| **历史记录保存方式**| 拼接在 `agent_scratchpad` 的长文本中 | 独立的 `ToolMessage` 消息块（强依赖 `tool_call_id`） |
| **并发工具能力** | 不支持（一次只能思考和调用一个工具） | **完全支持**（Parallel Tool Calling，可一次返回多个指令） |
| **LangChain 创建函数**| `create_react_agent` | `create_tool_calling_agent` |

---

## 💡 四、 架构师选型最佳实践

1. **首选 Function Calling**：只要你接入的大模型（如 GPT-4, Claude-3.5, Qwen-Max/Plus 等较新版本）原生支持工具调用接口，**请毫无保留地选择 Function Calling 模式**。它是商业级应用确保高可用性（SLA）的基础。
2. **ReAct 的退路场景**：只有当你被迫使用**较小的开源模型**（如 7B 级别且未经 Tool 微调的模型），或者业务场景极其特殊，要求**必须将 AI 的思考过程展示给 C 端用户看**时，才考虑使用 ReAct 模式。

# 多工具的使用

## 使用ReAct

In [ ]:
import os
import dotenv

# 导入 Tavily 搜索引擎（专为 Agent 设计的无广告、结构化搜索引擎）
from langchain_community.tools.tavily_search import TavilySearchResults

# 导入 LangChain 的标准工具封装类
from langchain.tools import Tool

# 导入 OpenAI/兼容大模型接口
from langchain_openai import ChatOpenAI

# 导入 LangChain Hub（类似于 Docker Hub，专门用来拉取云端的优秀 Prompt 模板）
from langchain import hub

# 导入现代版 Agent 创建函数和物理执行器
from langchain.agents import create_react_agent, AgentExecutor

# ==============================================================================
# ⚙️ 准备工作：环境变量加载
# ==============================================================================
dotenv.load_dotenv(override=True)
# 确保在 .env 文件中已经配置了 TAVILY_API_KEY，否则工具无法联网
# 防止你的 API 请求被本地的代理软件（如 Clash、VPN）拦截而导致网络连接报错（SSLError 等）。
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
# 1. 基础配置
# 2. 定义严格的大模型 (调用工具温度务必设为 0)
dotenv.load_dotenv(
    # 在工程根目录只保留一个 .env，所有模块都引用它。这是最不容易出错的方式。
    override=True,  # 如果加载.env文件出现了同名环境变脸，会进行覆盖
    # dotenv_path=".env",  # 可以只当需要加载的路径
)

# ==============================================================================
# 🔍 步骤 1：获取 Tavily 搜索工具的实例
# ==============================================================================
# 参数: max_results=3
# 作用: 限制搜索引擎每次最多返回 3 条网页摘要。
# 机制: 防止搜索结果太长导致大模型的 Token 数量超载（也就是爆显存）。
tavily_search = TavilySearchResults(max_results=3)

# ==============================================================================
# 🛠️ 步骤 2：获取一个标准的搜索工具 (Tool)
# ==============================================================================
# 机制: 大模型不认识 TavilySearchResults 对象，我们必须用 Tool 类把它“包装”成标准工具。
search_tool = Tool(
    # 参数: name (工具名称)
    # 作用: 大模型在输出 Action 时，必须严格原样输出这个名字。
    name="Search",
    # 参数: func (执行函数)
    # 作用: 告诉底层代码，当大模型决定用这个工具时，去执行哪个具体的 Python 函数。
    func=tavily_search.run,
    # 参数: description (工具描述) 🌟【极度重要】
    # 作用: 这绝对不是写给程序员看的注释！这是直接发给大模型看的“工具使用说明书”。
    # 机制: 大模型会阅读这段话，来判断当前用户的问题适不适合用这个工具。
    description="用于检索互联网上的信息，尤其是实时新闻、天气情况或未知的客观事实。",
)

# 构建工具箱（虽然现在只有1个工具，但未来你可以塞进去计算器、日历等多个工具）
tools = [search_tool]

# ==============================================================================
# 🧠 步骤 3：获取大语言模型 (LLM)
# ==============================================================================
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",  # 确保模型名称准确
    # 参数: temperature=0 🌟【ReAct 模式铁律】
    # 作用: 消除大模型的“想象力和创造力”。
    # 机制: ReAct 模式要求大模型必须严格按照极其死板的格式输出文字，一旦它发散思维随便乱说，
    #       底层的正则提取器就会崩溃。所以必须设为 0，让它变成严谨的机器！
    temperature=0,
)

# ==============================================================================
# 🧩 步骤 4：获取 Agent (创建思考大脑)
# ==============================================================================
# 参数: "hwchase17/react"
# 作用: 从云端拉取一套官方写好的、极其严苛的纯英文 Prompt 模板。
# 机制: 这个模板里写满了规则，强制大模型必须按照以下格式思考：
#       Thought: (我现在的想法是什么)
#       Action: (我要用什么工具，必须从 [Search, Calculator] 中选)
#       Action Input: (我要传给工具的参数是什么)
prompt = hub.pull("hwchase17/react")
print("prompt", prompt)
# 调用专属工厂函数，将 大模型、工具箱说明书、思考规则模板 揉合在一起，诞生“思考大脑”
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# ==============================================================================
# ⚙️ 步骤 5：获取 AgentExecutor 的实例 (创建物理执行器)
# ==============================================================================
# 机制: Agent 只负责“想”，AgentExecutor 负责拿着想法去“做”（在本地运行代码）。
agent_executor = AgentExecutor(
    agent=agent,  # 塞入刚刚创建的大脑
    tools=tools,  # 再次塞入工具箱（因为此时需要在本地真正运行那个 func 函数）
    # 参数: verbose=True
    # 作用: 开启上帝视角。你将在控制台看到绿色的 Thought 和蓝色的 Action，非常震撼。
    verbose=True,
    # 参数: handle_parsing_errors=True 🌟【生产环境救命稻草】
    # 作用: 自动纠错与重试机制。
    # 机制: 在 ReAct 模式下，如果大模型忘了写 "Action:" 这几个字，系统会报错。
    #       设为 True 后，系统不会崩溃，而是会自动把报错信息扔回给大模型：“你刚才输出的格式不对，请重写！”
    handle_parsing_errors=True,
    # 参数: max_iterations=6
    # 作用: 限制大模型最多只能连续进行 6 次“思考-行动”循环。
    # 机制: 防止大模型遇到解不开的难题时，陷入“无限死循环搜索”，把你的 API 余额瞬间刷光。
    max_iterations=6,
)

# ==============================================================================
# 🚀 步骤 6：通过 AgentExecutor 发起真实的调用
# ==============================================================================
print("\n🚀 正在唤醒 ReAct Agent 特工...\n")

# 注意: invoke 必须接收一个字典，其中 "input" 这个 Key 对应用户的提问。
result = agent_executor.invoke(
    {"input": "请帮我查一下昨天关于 LangChain 的最新新闻是什么？"}
)

print("\n" + "=" * 50)
# 提取出最后生成的最终回复
print(f"🎯 最终总结结果：\n{result['output']}")
print("=" * 50)

## 使用Function_Calling

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

# 🌟 新增导入：PythonREPL，LangChain封装的Python执行环境，常用于复杂数学计算
from langchain_experimental.utilities.python import PythonREPL
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor

# ==============================================================================
# ⚙️ 准备工作
# ==============================================================================
dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

# ==============================================================================
# 🛠️ 步骤 1：打造“百宝箱” (定义多个工具)
# ==============================================================================

# 🔧 工具 A：联网搜索工具
search = TavilySearchResults(max_results=3)
search_tool = Tool(
    name="Search",
    func=search.run,
    # 🚨 核心要点：在多工具场景下，description 是大模型“路由”的唯一依据！
    description="用于搜索互联网上的信息，特别是股票价格和新闻。",  #
)

# 🔧 工具 B：数学计算工具
# 使用 PythonREPL 作为计算器 [cite: 2523]
python_repl = PythonREPL()
calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如计算百分比变化。",  #
)

# 🎒 将两个工具一并放入工具箱 [cite: 2675]
tools = [search_tool, calc_tool]

# ==============================================================================
# 🧠 步骤 2：初始化大模型与极简提示词
# ==============================================================================
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
    temperature=0,  # 多工具选择极度考验逻辑，必须为0
)

# Function Calling 模式的极简提示词
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一位专业的金融分析特工。请利用手中的工具分步骤解决用户的问题。",
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

# ==============================================================================
# 🧩 步骤 3：组装特工并执行
# ==============================================================================
# 现代写法：自动绑定 tools 列表中的所有工具
agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # 开启上帝视角，观察大模型如何在两个工具间切换
    handle_parsing_errors=True,
)

print("\n🚀 正在唤醒双核金融特工...\n")

# 提出复合查询 [cite: 2686]
query = "特斯拉当前股价是多少？比去年上涨了百分之几？"  # [cite: 2686]
result = agent_executor.invoke({"input": query})

print("\n" + "=" * 50)
print(f"🎯 最终分析报告：\n{result['output']}")
print("=" * 50)

In [2]:
import os
import dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_tavily import TavilySearch  # 新版 Tavily 工具
from langchain_experimental.utilities import PythonREPL
from langchain.tools import tool

# ✅ 注意：AgentExecutor 来自 langchain_classic
from langchain_classic.agents import AgentExecutor, create_openai_tools_agent


dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

# ---- tools ----
search_tool = TavilySearch(max_results=3, topic="finance")

python_repl = PythonREPL()


@tool
def calculator(code: str) -> str:
    """执行 Python 计算；请用 print(...) 输出结果。"""
    return python_repl.run(code)


tools = [search_tool, calculator]

# ---- llm ----
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
    temperature=0,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一位专业的金融分析特工。请利用手中的工具分步骤解决用户的问题。",
        ),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

query = "特斯拉当前股价是多少？比去年上涨了百分之几？"
result = agent_executor.invoke({"input": query})
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': '特斯拉当前股价是多少？比去年上涨了百分之几？', 'topic': 'finance'}`


{'query': '特斯拉当前股价是多少？比去年上涨了百分之几？', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://hk.finance.yahoo.com/news/%E7%89%B9%E6%96%AF%E6%8B%89%E5%85%AC%E4%BD%88%E5%AD%A3%E7%B8%BE%E8%A6%81%E7%9C%8B%E7%9A%84%E4%B8%89%E5%A4%A7%E9%87%8D%E9%BB%9E-150541499.html', 'title': '特斯拉公佈季績要看的三大重點 - Yahoo 財經', 'content': '對於特斯拉(NASDAQ：TSLA)股東來說，這是瘋狂的一年。因為股價今年以來上漲了400％以上。在過去的一年中，股價上漲了近800％。', 'score': 0.99988043, 'raw_content': None}, {'url': 'https://hk.finance.yahoo.com/news/90-%E6%BC%B2%E5%B9%85%E5%A4%AA%E8%B6%85%E9%81%8E-%E7%89%B9%E6%96%AF%E6%8B%89%E9%81%AD%E7%BE%8E%E9%8A%80%E9%99%8D%E8%A9%95-124003957.html', 'title': '90%漲幅太超過？特斯拉遭美銀降評 - Yahoo 財經', 'content': '不過特斯拉股價已連續十天上漲，今年一次漲幅是去年多次漲幅累計的九倍。股價從52 週來最低點上漲約300%，並且今年迄今為止累計上漲約70%，這樣的收益壓過標', 'score': 0.99978846, 'raw_content': None}, {'url': 'https://hk.finance.yahoo.com/news/%E7%89%B

# create_agent + LangGraph 版本（等价重构，可直接用）

In [ ]:
# create_agent + LangGraph 版本（等价重构，可直接用）
import os
import dotenv

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from langchain_tavily import TavilySearch
from langchain_experimental.utilities import PythonREPL
from langchain.tools import tool


# ==============================================================================
# ⚙️ 准备工作
# ==============================================================================
dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

# ==============================================================================
# 🛠️ Tools
# ==============================================================================
# Tool A: Tavily 搜索（金融）
search_tool = TavilySearch(max_results=5, topic="finance")

# Tool B: Python 计算器
python_repl = PythonREPL()


@tool
def calculator(code: str) -> str:
    """执行 Python 代码做计算。注意：如果你要返回值，请用 print(...) 输出。"""
    return python_repl.run(code)


tools = [search_tool, calculator]

# ==============================================================================
# 🧠 Model
# ==============================================================================
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
    temperature=0,
)

# ==============================================================================
# 🧩 Agent（create_agent = LangGraph runtime）
# ==============================================================================
system_prompt = (
    "你是一位专业的金融分析特工。请严格按步骤使用工具解决问题：\n"
    "1) 用 Search 获取“特斯拉 TSLA 当前股价”（注明来源与时间点）；\n"
    "2) 用 Search 获取“去年同一天（或尽量接近的交易日）的 TSLA 股价”；\n"
    "3) 用 Calculator 计算涨跌幅：(当前-去年)/去年*100，并给出结果（保留两位小数）；\n"
    "4) 如果搜索结果存在多个价格口径（盘中/收盘/不同市场），优先使用‘收盘价’；若只能拿到盘中价，请说明。\n"
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    name="finance_agent",
)

# ==============================================================================
# 🚀 调用（传入 messages 更新 state）
# ==============================================================================
query = "特斯拉当前股价是多少？比去年上涨了百分之几？"

result = agent.invoke({"messages": [{"role": "user", "content": query}]})

# result 是 state（包含 messages 等），最后一条通常是最终回答
final_msg = result["messages"][-1]
print(final_msg.content)

In [ ]:
import os
import dotenv

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_experimental.utilities import PythonREPL
from langchain.tools import tool

# ✅ 显式 LangGraph
from langgraph.prebuilt import create_react_agent


dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

# --- tools ---
search_tool = TavilySearch(max_results=5, topic="finance")

python_repl = PythonREPL()


@tool
def calculator(code: str) -> str:
    """执行 Python 代码做计算；要返回值请 print(...)。"""
    return python_repl.run(code)


tools = [search_tool, calculator]

# --- model ---
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
    temperature=0,
)

system_prompt = (
    "你是一位专业的金融分析特工。请严格按步骤使用工具解决问题：\n"
    "1) 用 Search 获取 TSLA 当前股价（注明来源与时间点）；\n"
    "2) 用 Search 获取去年同日（或最近交易日）的 TSLA 股价；\n"
    "3) 用 Calculator 计算涨跌幅：(当前-去年)/去年*100，保留两位小数；\n"
    "4) 优先使用收盘价；若只能拿到盘中价，请说明。\n"
)

# ✅ 这里返回的就是 LangGraph graph
graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,  # 有的版本叫 prompt / state_modifier，按你版本为准
)

query = "特斯拉当前股价是多少？比去年上涨了百分之几？"

state = graph.invoke({"messages": [{"role": "user", "content": query}]})
print(state["messages"][-1].content)